# NER Performance Evaluation Pipeline

In [1]:
'''
Function: ner_zero_shot
Description:
 - Performs zero-shot Named Entity Recognition (NER) using the GPT-4o model.
 - Takes instructions, input text, model name, and temperature value as parameters.
 - Sends the instructions and the text to the OpenAI chat API.
 - Receives and cleans the model's response.
 - Attempts to parse the response as JSON.
 - Returns the parsed output if valid, otherwise returns None.
'''

from openai import OpenAI
import json

def ner_zero_shot(instruction, paragraph, model, temp):
    client = OpenAI()
    user_query = 'TEXT: {paragraph}'
    
    response = client.chat.completions.create(
      model = model,
      temperature = temp,
      messages = [
        {'role': 'system', 'content': instruction},
        {'role': 'user', 'content': user_query.format(paragraph=paragraph)}
      ]
    )

    content = response.choices[0].message.content
    cleaned_content = content.strip('```python\n').strip('```')
    
    try:
        output = json.loads(cleaned_content)
        return output
    except json.JSONDecodeError as e:
        return None
    

In [2]:
'''
Function: evaluate_distinct_entities
Description:
 - Evaluates Named Entity Recognition (NER) performance at both paragraph and document levels.
 - Takes labels, paragraphs, predicted and gold standard entities from corresponding paragraphs, 
   and download location as parameters.
 - Calculates precision, recall, and F1-score for each label in each paragraph.
 - Aggregates entity sets across paragraphs to compute overall metrics per label.
 - Saves detailed outputs including:
   - A .txt file logging paragraph-level predictions and gold terms.
   - An Excel file for paragraph-level performance.
   - An Excel file for overall (document-level) performance.
'''

# from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from evaluation_variables_t1r1 import labels, paragraphs, gold_standard_terms, predicted_terms

# NEED PYTHON 3.9 OR MORE FOR TYPE DESCRIPTION
# def evaluate_distinct_entity(
#     label: list[str],
#     paragraph: list[str],
#     gold_entity: list[dict[str, list[str]]],
#     pred_entity: list[dict[str, list[str]]],
#     dir_path: str
# ) -> None:
    
def evaluate_distinct_entity(label: list, paragraph: list, gold_entity: list, pred_entity: list, dir_path: str) -> None:
    
    # paragraph-level performance calculation 
    # variable declaration
    all_gold_ent = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    all_pred_ent = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    df_score_para = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_para = []
    eval_value_para = []
    
    # retrieving paragraph, gold standard terms, and predicted terms from zipped list
    for para, gold_ent, pred_ent in zip(paragraph, gold_entity, pred_entity):
        para_index = paragraph.index(para)
        
        # writing paragraph number, paragraph, gold standard terms, and predicted terms in a text file
        with open(f'{dir_path}\\paragraph-with-all-entity.txt', 'a', encoding='utf-8') as file:
            file.write(f'PARAGRAPH NUMBER: {para_index}\n')
            file.write(f'PARAGRAPH: {para}\n')
            file.write(f'GOLD ENTITY: {gold_ent}\n')
            file.write(f'PRED ENTITY: {pred_ent}\n')
            file.write('=======================================================\n')
        
        print(f'Updated paragraph-with-all-entity.txt => {dir_path}')
        
        # retrieving labels from list
        for l in label:
            
            # creating entity set with unique entities
            unique_gold_ent = set(gold_ent[l])
            unique_pred_ent = set(pred_ent[l])
            
            # storing entities (label-wise) for document-level calculation
            all_gold_ent[l].update(unique_gold_ent)
            all_pred_ent[l].update(unique_pred_ent)
            
            # calculate confusion matrix
            tp = unique_gold_ent & unique_pred_ent
            fp = unique_pred_ent - unique_gold_ent
            fn = unique_gold_ent - unique_pred_ent
            
            # calculate precision, recall and f1-score for each paragraph
            precision = len(tp) / (len(tp) + len(fp)) if unique_pred_ent else 0
            recall = len(tp) / (len(tp) + len(fn)) if unique_gold_ent else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # storing performance for each paragraph in list
            eval_metric_para.extend([f'{para_index} {l.upper()} Precision', 
                                     f'{para_index} {l.upper()} Recall', 
                                     f'{para_index} {l.upper()} F1'])

            eval_value_para.extend([f'{precision:.2f}',
                                    f'{recall:.2f}', 
                                    f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_para['metric'] = eval_metric_para
    df_score_para['value'] = eval_value_para

    # downloading paragraph-level performace in a spreadsheet
    df_score_para.to_excel(f'{dir_path}\\score-para-DIS-ENT.xlsx', index=False)
    print(f'Downloaded score-para-DIS-ENT.xlsx => {dir_path}')
    
    # document-level performance calculation  
    # variable declaration
    df_score_doc = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_doc = []
    eval_value_doc = []
    
    # retrieving label from list
    for l in label:

        # calculate confusion matrix
        tp = all_gold_ent[l] & all_pred_ent[l]
        fp = all_pred_ent[l] - all_gold_ent[l]
        fn = all_gold_ent[l] - all_pred_ent[l]
        
        # calculate precision, recall and f1-score for entire document
        precision = len(tp) / (len(tp) + len(fp)) if all_pred_ent[l] else 0
        recall = len(tp) / (len(tp) + len(fn)) if all_gold_ent[l] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{l}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # storing performance for the entire document in list
        eval_metric_doc.extend([f'{l.upper()} (Overall) Precision', 
                                f'{l.upper()} (Overall) Recall', 
                                f'{l.upper()} (Overall) F1'])
        
        eval_value_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])

    # creating dataframe from list
    df_score_doc['metric'] = eval_metric_doc
    df_score_doc['value'] = eval_value_doc

    # downloading document-level performace in a spreadsheet
    df_score_doc.to_excel(f'{dir_path}\\score-doc-DIS-ENT.xlsx', index=False)
    print(f'Downloaded score-doc-DIS-ENT.xlsx => {dir_path}')
    
#     # document-level analysis
#     # overlap between chemical and material entities
#     print('Overlap between chemical and material entities:')
#     overlap_in_gold_standard = len(all_gold_ent['chemical'] & all_gold_ent['material'])
#     overlap_in_predicted = len(all_pred_ent['chemical'] & all_pred_ent['material'])
#     print(f'In gold standard data: {overlap_in_gold_standard}')
#     print(f'In predicted data: {overlap_in_predicted}')
    
#     # overlap between chemical and structure entities
#     print('Overlap between chemical and structure entities:')
#     overlap_in_gold_standard = len(all_gold_ent['chemical'] & all_gold_ent['structure'])
#     overlap_in_predicted = len(all_pred_ent['chemical'] & all_pred_ent['structure'])
#     print(f'In gold standard data: {overlap_in_gold_standard}')
#     print(f'In predicted data: {overlap_in_predicted}')
    
    
#     # overlap between material and structure entities
#     print('Overlap between chemical and structure entities:')
#     overlap_in_gold_standard = len(all_gold_ent['material'] & all_gold_ent['structure'])
#     overlap_in_predicted = len(all_pred_ent['material'] & all_pred_ent['structure'])
#     print(f'In gold standard data: {overlap_in_gold_standard}')
#     print(f'In predicted data: {overlap_in_predicted}')
    
#     print(all_gold_ent['chemical'] & all_gold_ent['material'], all_pred_ent['chemical'] & all_pred_ent['material'])
#     print(all_gold_ent['chemical'] & all_gold_ent['structure'], all_pred_ent['chemical'] & all_pred_ent['structure'])
#     print(all_gold_ent['material'] & all_gold_ent['structure'], all_pred_ent['material'] & all_pred_ent['structure'])
    

In [3]:
'''
Function: evaluate_all_entities
Description:
 - Evaluates Named Entity Recognition (NER) performance at both paragraph and document levels.
 - Accepts predicted terms, gold standard terms, and corresponding paragraphs for comparison.
 - Calculates precision, recall, and F1-score for each label in each paragraph.
 - Aggregates entity sets across paragraphs to compute overall metrics per label.
 - Saves detailed outputs including:
   - A .txt file logging paragraph-level predictions and gold terms.
   - An Excel file for paragraph-level performance.
   - An Excel file for overall (document-level) performance.
'''

# from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from collections import Counter

def evaluate_all_entity(label: list, paragraph: list, gold_entity: list, pred_entity: list, dir_path: str) -> None:
    
    # paragraph-level performance calculation 
    # variable declaration
    all_gold_ent = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    all_pred_ent = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    df_score_para = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_para = []
    eval_value_para = []
    
    # retrieving paragraph, gold standard terms, and predicted terms from zipped list
    for para, gold_ent, pred_ent in zip(paragraph, gold_entity, pred_entity):
        para_index = paragraph.index(para)
        
        # writing paragraph number, paragraph, gold standard terms, and predicted terms in a text file
#         with open(f'{dir_path}\\paragraph-with-all-entity.txt', 'a', encoding='utf-8') as file:
#             file.write(f'PARAGRAPH NUMBER: {para_index}\n')
#             file.write(f'PARAGRAPH: {para}\n')
#             file.write(f'GOLD ENTITY: {gold_ent}\n')
#             file.write(f'PRED ENTITY: {pred_ent}\n')
#             file.write('=======================================================\n')
        
#         print(f'Updated paragraph-with-all-entity.txt => {dir_path}')
        
        # retrieving labels from list
        for l in label:
            
            # count occurrence of each term
            gold_counter = Counter(gold_ent[l])
            pred_counter = Counter(pred_ent[l])
            
            # storing entities (label-wise) for document-level calculation
            all_gold_ent[l].update(gold_counter)  # CHECK: IF SAME TERM COMES FROM 2ND PARAGRAPH
            all_pred_ent[l].update(pred_counter)
            
            # calculate confusion matrix
            tp_counter = gold_counter & pred_counter  # Intersection of counts
            tp_sum = sum(tp_counter.values())

            fp_counter = pred_counter - gold_counter  # Predicted but not in gold
            fp_sum = sum(fp_counter.values())

            fn_counter = gold_counter - pred_counter  # Gold but not in predicted
            fn_sum = sum(fn_counter.values())

            # calculate precision, recall and f1-score for each paragraph
            precision = tp_sum / (tp_sum + fp_sum) if pred_counter else 0
            recall = tp_sum / (tp_sum + fn_sum) if gold_counter else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # storing performance for each paragraph in list
            eval_metric_para.extend([f'{para_index} {l.upper()} Precision', 
                                     f'{para_index} {l.upper()} Recall', 
                                     f'{para_index} {l.upper()} F1'])

            eval_value_para.extend([f'{precision:.2f}',
                                    f'{recall:.2f}', 
                                    f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_para['metric'] = eval_metric_para
    df_score_para['value'] = eval_value_para

    # downloading paragraph-level performace in a spreadsheet
    df_score_para.to_excel(f'{dir_path}\\score-para-ALL-ENT.xlsx', index=False)
    print(f'Downloaded score-para-ALL-ENT.xlsx => {dir_path}')

    # document-level performance calculation  
    # variable declaration
    df_score_doc = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_doc = []
    eval_value_doc = []
    
    # ADDED FOR OVERLAP
    tp_sum_all_labels, fp_sum_all_labels, fn_sum_all_labels = 0, 0, 0
    
    # retrieving label from list
    for l in label:

        # calculate confusion matrix
        tp_counter = all_gold_ent[l] & all_pred_ent[l]
        tp_sum = sum(tp_counter.values())

        fp_counter = all_pred_ent[l] - all_gold_ent[l]
        fp_sum = sum(fp_counter.values())

        fn_counter = all_gold_ent[l] - all_pred_ent[l]
        fn_sum = sum(fn_counter.values())
        
        # calculate precision, recall and f1-score for entire document
        precision = tp_sum / (tp_sum + fp_sum) if all_pred_ent[l] else 0
        recall = tp_sum / (tp_sum + fn_sum) if all_gold_ent[l] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{l}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # storing performance for the entire document in list
        eval_metric_doc.extend([f'{l.upper()} (Overall) Precision', 
                                f'{l.upper()} (Overall) Recall', 
                                f'{l.upper()} (Overall) F1'])
        
        eval_value_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_doc['metric'] = eval_metric_doc
    df_score_doc['value'] = eval_value_doc

    # downloading document-level performace in a spreadsheet
    df_score_doc.to_excel(f'{dir_path}\\score-doc-ALL-ENT.xlsx', index=False)
    print(f'Downloaded score-doc-ALL-ENT.xlsx => {dir_path}')
       

## *Main:* (for performance evaluation)

#### <span style="color: blue;">Description:</span>
- calls 
    - ***ner_zero_shot()*** 
    - ***evaluate_distinct_entities()***
    - ***evaluate_all_entities()***
- generates files through *performance_metrics()*
    - *dl-all-terms.txt*
    - *dl-performance-paragraph.xlsx*
    - *dl-performance-overall.xlsx*
- generates files *evaluation_variables.py*

#### <span style="color: red;">Check:</span>
- *file_path* for evaluation data
- *labels* in case of merging or changing labels
- argument value (to set temperature) in *annotator_without_examples()* calling
- *download_location* for saving files

In [4]:
'''
Script Description:
 - Reads NER evaluation data from a text file.
 - Separates paragraphs and gold standard terms.
 - Parses the gold terms from JSON strings into dictionaries.
 - Performs zero-shot NER prediction using GPT-4o for each paragraph.
 - Ensures that predictions include all expected labels.
 - Saves evaluation variables to a file for reproducibility.
 - Evaluates model performance using a custom evaluation function.
'''
import json

def process_input_data(text_file):
    # read input data from text file
    with open(text_file, 'r', encoding='utf-8') as file:
        list_ = file.read().splitlines()

    # store paragraphs and annotations in different lists
    paragraph = []
    gold_ent_str = []

    for item in list_:
        if list_.index(item) == 0 or list_.index(item) % 2 == 0:
            paragraph.append(item)
        else:
            gold_ent_str.append(item)

    # convert annotations (in string format) to nested object
    gold_entity = []

    for item in gold_ent_str:
        try:
            json_obj = json.loads(item)              # Convert to dictionary
            gold_entity.append(json_obj)  # Add to list of dictionaries
        except json.JSONDecodeError as e:
            print(f'Error decoding JSON for item: {item}\nError: {e}')

    return paragraph, gold_entity


def annotate(instruction, paragraph, label, gold_entity, model, temp, dir_path):
    # predict entities from each paragraph for every instruction
    pred_entity = []
    for para in paragraph:
        pred_ent_in_para = dict()

        for inst in instruction:
            response = ner_zero_shot(inst, para, model, temp)    # calling annotator()
            if response:
                llm_label = list(response.keys())
                llm_ent = list(response.values())
                pred_ent_in_para[llm_label[0]] = llm_ent[0]
    #         else:
    #             print("ERROR::", response)
        
        # check for missing labels in predicted entities
        # all labels should be there even if they do not have any entities 
        if len(pred_ent_in_para) < len(label):    
            print('Label missing in predicted data.')            
            revised_data = dict()
            
            for l in label:
                if l not in pred_ent_in_para:
                    pred_ent_in_para[l] = []
                    print(f'Label -- {l} -- added to predicted data.')
            
            # organize the annotations' labels according to label's order
            for l in label:
                revised_data[l] = pred_ent_in_para[l]
                
            pred_ent_in_para = revised_data
        
        pred_entity.append(pred_ent_in_para)

    # save label, paragraph, gold_entity, pred_entity variables
    with open(f'{dir_path}\\evaluation_variable.py', 'w', encoding='utf-8') as file:
        file.write('label = ' + repr(label) + '\n')
        file.write('paragraph = ' + repr(paragraph) + '\n')
        file.write('gold_entity = ' + repr(gold_entity) + '\n')
        file.write('pred_entity = ' + repr(pred_entity) + '\n')
        
    print(f'Downloaded evaluation_variable.py => {dir_path}')
    
    return pred_entity


In [6]:
from chatgpt_prompt import instruction

label = [
    'chemical',
    'material',
    'structure',
    'property',
    'application',
    'process',
    'equipment',
    'measurement',
    'abbreviation'
]

paragraph, gold_ent = process_input_data(text_file='sample-eval-data.txt')

pred_ent = annotate(
    instruction=instruction,
    paragraph=paragraph,
    label=label,
    gold_entity=gold_ent,
    model='gpt-4o',
    temp=0.2,
    dir_path='output/zero-shot'
)

evaluate_distinct_entity(
    label=label,
    paragraph=paragraph,
    gold_entity=gold_ent,
    pred_entity=pred_ent,
    dir_path='output/zero-shot'
)

evaluate_all_entity(
    label=label,
    paragraph=paragraph,
    gold_entity=gold_ent,
    pred_entity=pred_ent,
    dir_path='output/zero-shot'
)

Downloaded evaluation_variable.py => output/zero-shot
Updated paragraph-with-all-entity.txt => output/zero-shot
Updated paragraph-with-all-entity.txt => output/zero-shot
Updated paragraph-with-all-entity.txt => output/zero-shot
Downloaded score-para-DIS-ENT.xlsx => output/zero-shot
chemical:	0.41(f) | 0.28(p) | 0.78(r)
material:	0.34(f) | 0.25(p) | 0.56(r)
structure:	0.65(f) | 0.68(p) | 0.63(r)
property:	0.24(f) | 0.19(p) | 0.33(r)
application:	0.00(f) | 0.00(p) | 0.00(r)
process:	0.67(f) | 1.00(p) | 0.50(r)
equipment:	0.00(f) | 0.00(p) | 0.00(r)
measurement:	0.75(f) | 0.60(p) | 1.00(r)
abbreviation:	0.00(f) | 0.00(p) | 0.00(r)
Downloaded score-doc-DIS-ENT.xlsx => output/zero-shot
Downloaded score-para-ALL-ENT.xlsx => output/zero-shot
chemical:	0.65(f) | 0.50(p) | 0.91(r)
material:	0.26(f) | 0.17(p) | 0.55(r)
structure:	0.64(f) | 0.64(p) | 0.64(r)
property:	0.20(f) | 0.15(p) | 0.29(r)
application:	0.00(f) | 0.00(p) | 0.00(r)
process:	0.62(f) | 1.00(p) | 0.45(r)
equipment:	0.00(f) | 0.0

# NEs Annotation Pipeline

In [6]:
## THIS FUNCTION IS ONLY NEEDED FOR CONTINUOUS ANNOTATION
import pandas as pd
def validate_entity_spans(terms_in_paragraph):
    columns = ['paragraph', 'start_index', 'end_index', 'label', 'llm_term', 'sliced_term']
    df = pd.DataFrame(terms_in_paragraph, columns=columns)
    df_sorted = df.sort_values(by="start_index")
    df_sorted.reset_index(drop=True, inplace=True)

    df_sorted['equal'] = df_sorted['llm_term'] == df_sorted['sliced_term']
    df_sorted['overlap'] = False

    for i in range(1, len(df_sorted)):
        if df_sorted.loc[i, 'start_index'] >= df_sorted.loc[i-1, 'start_index'] and \
            df_sorted.loc[i, 'start_index'] <= df_sorted.loc[i-1, 'end_index']:
            df_sorted.loc[i, 'overlap'] = True

    df_sorted = df_sorted[df_sorted["equal"] != False]
    df_sorted = df_sorted[df_sorted['overlap'] != True]
    # df_sorted = df_sorted[df_sorted['equal']]  # Keep only where equal is True
    # df_sorted = df_sorted[~df_sorted['overlap']]  # Remove rows with overlap

    # df_sorted

    output_list = []

    for index, row in df_sorted.iterrows():
        output_list.append([row['start_index'], row['end_index'], row['label']])

    final_output = {'entities': output_list}

    return final_output

In [3]:
# read evaluation data from text file and store paragraphs and terms in different list
def annotate(txt_file, label, inst, temp):

    with open(txt_file, 'r', encoding='utf-8') as file:
        paragraphs = file.read().splitlines()

    para_with_ents = []

    # predict terms from each paragraph for every instruction

    for para in paragraphs:
        paragraph_index = paragraphs.index(para)
        data = dict()

        for inst in instructions:
            response = ner_zero_shot(inst, para, temp)    # calling annotator()
            if response:
                label = list(response.keys())
                terms = list(response.values())
                data[label[0]] = terms[0]
    #         else:
    #             print("ERROR::", response)

        ents_in_para = []

        for key in data:
            end_index = 0

            for term in data[key]:
                term_length = len(term)
                start_index = para.find(term, end_index)

                if start_index != -1:
                    end_index = start_index + term_length
                    term_detail = [
                        paragraph_index,                    # Paragraph number within a file
                        start_index,                        # Starting position of a term
                        end_index,                          # Ending position of a term
                        key.upper(),                        # Label of a term
                        term,                               # Term extracted by llm
                        para[start_index:end_index]    # Term extracted using start and end position
                    ]

                    ents_in_para.append(term_detail)

        entities = validate_entity_spans(ents_in_para)   ## WHY ENTITY FORMATTER IS NEEDED
        para_with_ents.append([para, entities])

    spacy_annot = {'classes': labels, 'annotations': para_with_ents}
    
    return spacy_annot

    
def download_annotation(spacy_annot):
    # download annotation in spacy format
    download_location = ''
    with open('spacy_annotations.json', 'w', encoding='utf-8') as json_file:
        json.dump(spacy_annot, json_file, indent=4)

## *Main:* (for continuous annotation)

In [7]:
from chatgpt_prompt import instructions

labels = [
    'CHEMICAL',
    'MATERIAL',
    'STRUCTURE',
    'PROPERTY',
    'APPLICATION',
    'PROCESS',
    'EQUIPMENT',
    'MEASUREMENT',
    'ABBREVIATION'
]

spacy_annotations = annotate(txt_file='sample-text-data.txt', label=labels, inst=instructions, temp=0.2)
download_annotation(spacy_annotations)
